In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage

# -------------------------------------------------------
# 1. データ読み込み：OWID 食料グループ別カロリー供給
# -------------------------------------------------------
url = "https://ourworldindata.org/grapher/calorie-supply-by-food-group.csv"
df_raw = pd.read_csv(
    url,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

# 国別データだけ（Code が3文字）
df_raw = df_raw[df_raw["Code"].str.len() == 3]

# 各国の最新年を1行に
df_raw = df_raw.sort_values(["Entity", "Year"])
df_latest = df_raw.groupby("Entity").tail(1).set_index("Entity")

# 大陸データ（OWID の区分）を読み込み、国コードで結合
continent_url = "https://ourworldindata.org/grapher/continents-according-to-our-world-in-data.csv"
df_continents = pd.read_csv(
    continent_url,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)
df_continents = (
    df_continents.sort_values("Year")
    .drop_duplicates(subset="Code", keep="last")
    [["Code", "World region according to OWID"]]
    .rename(columns={"World region according to OWID": "Continent"})
)
df_latest = df_latest.join(
    df_continents.set_index("Code"),
    on="Code",
    how="left",
    validate="many_to_one",
)

# -------------------------------------------------------
# 2. CSV の列名を指定 → 分析用の短い列名に差し替え
# -------------------------------------------------------
food_cols = {
    "vegetal": "Plant foods",
    "meat": "Meat",
    "dairy": "Dairy",
    "eggs": "Eggs",
    "animal_fats": "Animal fats",
    "fish_seafood": "Fish and seafood",
}

selected_keys = ["vegetal", "meat", "dairy", "animal_fats", "fish_seafood"]
feature_cols = [food_cols[k] for k in selected_keys]

for col in feature_cols:
    if col not in df_latest.columns:
        raise ValueError(f"Column not found in df_latest: {col}")

df_features = df_latest[feature_cols].copy()

# 列名を短いもの（vegetal, meat, ...）に変更
df_features.columns = selected_keys

# 欠損を含む国は除外
df_features = df_features.dropna()

# 大陸名を確認しやすいよう、表示テーブルの先頭列に追加
df_features.insert(0, "Continent", df_latest["Continent"])

# データの表示
df_features

In [ ]:
from matplotlib.patches import Patch

# -------------------------------------------------------
# 3. 日本 + ランダム n_random カ国を選ぶ部分
# -------------------------------------------------------
np.random.seed(5)

n_random = 19 # 日本以外の国数指定ｓ

all_countries = df_features.index.tolist()
if "Japan" not in all_countries:
    raise ValueError("Japan not found in the dataset; check the dataset format.")

other_countries = [c for c in all_countries if c != "Japan"]

if len(other_countries) < n_random:
    raise ValueError("Not enough countries with complete data to sample from.")

sampled_countries = np.random.choice(other_countries, size=n_random, replace=False)
selected_countries = ["Japan"] + list(sampled_countries)

df_selected = df_features.loc[selected_countries]

# ★ ここで実際に何カ国選ばれているか確認 ★
print("\nNumber of selected countries:", len(df_selected))
print("Selected countries:")
print(df_selected.index.tolist())

# -------------------------------------------------------
# 4. z-score 標準化 → DataFrame に変換
# -------------------------------------------------------
# 大陸名は計算に含めず、食品群の数値列だけを使用
X = df_selected[selected_keys].to_numpy(dtype=float)
means = X.mean(axis=0)
stds = X.std(axis=0, ddof=0)
X_std = (X - means) / stds

df_std = pd.DataFrame(
    X_std,
    index=df_selected.index,
    columns=selected_keys,
)

# -------------------------------------------------------
# 5. 行方向のリンクage（ウォード法）
# -------------------------------------------------------
row_linkage = linkage(X_std, method="ward")

# -------------------------------------------------------
# 6. clustermap 描画（行数に応じて縦サイズを調整）
# -------------------------------------------------------
continent_colors = {
    "Africa": "#1b9e77",
    "Asia": "#d95f02",
    "Europe": "#7570b3",
    "North America": "#e7298a",
    "South America": "#66a61e",
    "Oceania": "#a6761d",
}

row_colors = ["red" if idx == "Japan" else "gray" for idx in df_std.index]

sns.set(context="notebook", style="white")

n_rows = df_std.shape[0]
fig_height = max(6, 0.3 * n_rows)  # 行数に応じて高さを調整

g = sns.clustermap(
    df_std,
    row_linkage=row_linkage,
    col_cluster=False,
    row_colors=row_colors,
    yticklabels=True,
    figsize=(8, fig_height),
)

g.ax_heatmap.set_xlabel("Food groups (standardized)")
g.ax_heatmap.set_ylabel("Countries")

# 並べ替え後の国名から大陸を参照し、ラベルの文字色を設定
for label in g.ax_heatmap.get_yticklabels():
    country = label.get_text()
    continent = df_selected.loc[country, "Continent"]
    label.set_color(continent_colors.get(continent, "gray"))

# 上部の空き領域に大陸ごとの色の凡例を表示
legend_handles = [
    Patch(facecolor=color, label=continent)
    for continent, color in continent_colors.items()
    if continent in df_selected["Continent"].values
]
g.ax_col_dendrogram.legend(
    handles=legend_handles,
    title="Continent",
    loc="center",
    ncol=2,
    frameon=False,
)

plt.show()